In [1]:
import os
import mne
import numpy as np
import pickle
import random
from sklearn.model_selection import train_test_split



In [ ]:
#This part of the code is just for previewing the data split based on seed number

# Paths for input EDF folders
increase_path = 'increase_edf_no400'
normal_path = 'normal_edf_no400'
output_folders = ['train', 'val', 'test']

# Create output folders if they don't exist
for folder in output_folders:
    os.makedirs(folder, exist_ok=True)
# Get list of all EDF files for both classes
increase_files = [f for f in os.listdir(increase_path) if f.endswith('.edf')]
normal_files = [f for f in os.listdir(normal_path) if f.endswith('.edf')]
rs=42 #set seed
# Perform 80-10-10 split at the **file level** to avoid leakage
train_increase_files, temp_increase_files = train_test_split(increase_files, test_size=0.2, random_state=rs)
val_increase_files, test_increase_files = train_test_split(temp_increase_files, test_size=0.5, random_state=rs)

train_normal_files, temp_normal_files = train_test_split(normal_files, test_size=0.2, random_state=rs)
val_normal_files, test_normal_files = train_test_split(temp_normal_files, test_size=0.5, random_state=rs)


In [3]:
train_increase_files

['20.edf',
 '60.edf',
 '104.edf',
 '52.edf',
 '6.edf',
 '63.edf',
 '103.edf',
 '27.edf',
 '97.edf',
 '17.edf',
 '5.edf',
 '2.edf',
 '3.edf',
 '58.edf',
 '19.edf']

In [4]:
val_increase_files

['1.edf', '4.edf']

In [5]:
test_increase_files

['18.edf', '101.edf']

In [6]:
len(train_normal_files)

50

In [7]:
val_normal_files

['49.edf', '108.edf', '15.edf', '77.edf', '53.edf', '23.edf']

In [8]:
test_normal_files

['12.edf', '84.edf', '9.edf', '33.edf', '44.edf', '10.edf', '8.edf']

In [ ]:

# Paths for input EDF folders
increase_path = 'increase_edf_no400'
normal_path = 'normal_edf_no400'
#output_folders = ['train11', 'val11', 'test11']
#output_folders = ['train', 'val', 'test']
output_folders = ['train20', 'val20', 'test20']

# Create output folders if they don't exist
for folder in output_folders:
    os.makedirs(folder, exist_ok=True)

# Function to process EDF files: downsample, chunk, and label
def process_edf(file_path, label, sfreq=200, chunk_size=5, overlap=1, step_size=None,selected_channels=None):
    print(f"Processing: {file_path}")

    try:
        raw = mne.io.read_raw_edf(file_path, preload=True)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return []
    

    # Select only specified channels
    if selected_channels:
        raw.pick_channels(selected_channels)

    # Downsample if necessary
    raw._data *= 1e6  # Convert to microvolts for consistency
    if raw.info['sfreq'] != sfreq:
        raw.resample(sfreq)

    data = raw.get_data()  # (n_channels, n_samples)
    n_channels, n_samples = data.shape

    chunk_samples = int(chunk_size * sfreq)  # Number of samples per 5s chunk
    overlap_samples = int(overlap * sfreq)   # Number of overlapping samples

    # Use specified step size, or calculate it based on overlap
    if step_size is None:
        step_size = chunk_samples - overlap_samples

    chunks = []

    # Generate overlapping chunks
    for start in range(0, n_samples - chunk_samples + 1, step_size):
        chunk_data = data[:, start:start + chunk_samples]
        chunks.append((chunk_data, label))  # Store chunk and label as a tuple

    return chunks

# Get list of all EDF files for both classes
increase_files = [f for f in os.listdir(increase_path) if f.endswith('.edf')]
normal_files = [f for f in os.listdir(normal_path) if f.endswith('.edf')]

# Perform 80-10-10 split at the **file level** to avoid leakage
rs=42 #seed number for data split
train_increase_files, temp_increase_files = train_test_split(increase_files, test_size=0.2, random_state=rs)
val_increase_files, test_increase_files = train_test_split(temp_increase_files, test_size=0.5, random_state=rs)

train_normal_files, temp_normal_files = train_test_split(normal_files, test_size=0.2, random_state=rs)
val_normal_files, test_normal_files = train_test_split(temp_normal_files, test_size=0.5, random_state=rs)

# Helper function to process a list of files
def process_files(file_list, label, step_size,selected_channels=None):
    chunks = []
    for filename in file_list:
        file_path = os.path.join(increase_path if label == 1 else normal_path, filename)
        chunks.extend(process_edf(file_path, label, step_size=step_size, selected_channels=selected_channels))
    return chunks

# Process files for each split
increase_step_size = 250  # 3.75 seconds overlap for increase class
normal_step_size = 1000   # No overlap for normal class


#select a subset of channels,
#subset_channels = ['FP1','FP2','F3','FZ','F4','FC3','FCZ','FC4','C3','CZ','C4']
#subset_channels = ['FP1', 'FP2', 'F7', 'F3', 'FZ', 'F4', 'F8', 'FT7', 'FC3', 'FCZ', 'FC4', 'FT8', 'T3', 'C3', 'CZ', 'C4', 'T4', 'TP7', 'CP3', 'CPZ', 'CP4', 'TP8', 'T5', 'P3', 'PZ', 'P4', 'T6', 'O1', 'OZ', 'O2']
subset_channels=['FP1','F7','F3','F8','FZ','FC4', 'FT8', 'T3', 'C3', 'CZ', 'T4', 'TP7', 'CP3', 'CPZ', 'CP4','T5', 'P3', 'PZ', 'P4', 'T6']
train_increase = process_files(train_increase_files, label=1, step_size=increase_step_size,selected_channels=subset_channels)
val_increase = process_files(val_increase_files, label=1, step_size=increase_step_size,selected_channels=subset_channels)
test_increase = process_files(test_increase_files, label=1, step_size=increase_step_size,selected_channels=subset_channels)

train_normal = process_files(train_normal_files, label=0, step_size=normal_step_size,selected_channels=subset_channels)
val_normal = process_files(val_normal_files, label=0, step_size=normal_step_size,selected_channels=subset_channels)
test_normal = process_files(test_normal_files, label=0, step_size=normal_step_size,selected_channels=subset_channels)


train_chunks = train_increase + train_normal
val_chunks = val_increase + val_normal
test_chunks = test_increase + test_normal

# # Ensure balanced datasets by truncating to the smaller size
# min_train_length = min(len(train_increase), len(train_normal))
# train_chunks = train_increase[:min_train_length] + train_normal[:min_train_length]

# min_val_length = min(len(val_increase), len(val_normal))
# val_chunks = val_increase[:min_val_length] + val_normal[:min_val_length]

# min_test_length = min(len(test_increase), len(test_normal))
# test_chunks = test_increase[:min_test_length] + test_normal[:min_test_length]

# Shuffle the data to avoid ordering bias
random.seed(rs)
random.shuffle(train_chunks)
random.shuffle(val_chunks)
random.shuffle(test_chunks)

# Function to save chunks as pickle files
def save_chunks(chunks, folder):
    for i, (chunk_data, label) in enumerate(chunks):
        filename = os.path.join(folder, f"chunk_{i}.pickle")
        with open(filename, 'wb') as f:
            pickle.dump({'X': chunk_data, 'y': label}, f)
        print(f"Saved: {filename}")

# Save chunks to train, val, and test folders
save_chunks(train_chunks, 'train20')
save_chunks(val_chunks, 'val20')
save_chunks(test_chunks, 'test20')

print("All files processed, split, and saved successfully!")


Processing: increase_edf_no400\20.edf
Extracting EDF parameters from c:\Users\wangs\Desktop\91_ASR\bp_refA_rmBadCh_ASR_ICrej_20230402_interpolated_corrected\increase_edf_no400\20.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 334999  =      0.000 ...   334.999 secs...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Processing: increase_edf_no400\60.edf
Extracting EDF parameters from c:\Users\wangs\Desktop\91_ASR\bp_refA_rmBadCh_ASR_ICrej_20230402_interpolated_corrected\increase_edf_no400\60.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 300999  =      0.000 ...   300.999 secs...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Processing: increase_edf_no400\104.edf
Extracting EDF parameters from c:\Users\wangs\Desktop\91_ASR\bp_refA_rmBadCh_ASR_ICrej_20230402_interpolated_corrected\increase_edf_no400\104.edf...


In [27]:
with open('test11/chunk_350.pickle', 'rb') as f:
    data = pickle.load(f)
    print(data)  # Should output: {'X': ..., 'y': ...}

{'X': array([[ 8.46182126,  5.47001732,  5.84054485, ..., -2.58743174,
         4.93830355,  1.26330308],
       [ 7.20173202,  5.57021429,  3.22135698, ..., -6.89847499,
        -0.38086176,  0.96565399],
       [-8.96629602, -4.82120486, -5.61853562, ..., -5.77256697,
        -3.90320218,  0.20324251],
       ...,
       [ 1.62571361,  3.23063682,  3.11118945, ..., -4.91534465,
        -0.9728878 ,  3.67840826],
       [-2.18336218,  1.63128147,  1.61322962, ..., -3.05617594,
        -0.12983786,  1.52408016],
       [-6.61356924, -0.60730145,  1.82123277, ..., -3.36768114,
         1.06175394,  2.64660017]]), 'y': 1}


In [28]:
data['X'].max()
print(len(data['X']))

11


In [ ]:
training_directory = 'test'  # Update this path to your training directory

def inspect_label_ratios(directory):
    total_labels = 0
    class_labels = {0: 0, 1: 0}  # Assuming binary labels: 0 for normal and 1 for increase

    for filename in os.listdir(directory):
        if filename.endswith('.pickle'):
            file_path = os.path.join(directory, filename)
            print(f"Processing {filename} from {directory}...")

            try:
                # Load the pickle file
                with open(file_path, 'rb') as f:
                    data_dict = pickle.load(f)

                # Check if the expected keys are in the dictionary
                if 'y' not in data_dict:
                    print(f"Warning: 'y' key not found in {filename}. Skipping this file.")
                    continue
                
                labels = data_dict['y']

                # Check the type of labels
                if isinstance(labels, list):
                    labels = np.array(labels)  # Convert list to numpy array
                elif isinstance(labels, int):
                    labels = np.array([labels])  # Convert single integer to a numpy array
                elif not isinstance(labels, np.ndarray):
                    print(f"Warning: Unexpected label format in {filename}. Skipping this file.")
                    continue

                # Count the labels
                total_labels += len(labels)
                class_labels[1] += np.sum(labels)  # Count positive class (increase)
                class_labels[0] += len(labels) - np.sum(labels)  # Count negative class (normal)

            except Exception as e:
                print(f"Error processing {filename}: {e}")

    # Calculate the ratios
    if total_labels > 0:
        increase_ratio = class_labels[1] / total_labels
        normal_ratio = class_labels[0] / total_labels
    else:
        increase_ratio = 0
        normal_ratio = 0

    print(f"\nLabel Ratios:")
    print(f"Increase Class Ratio: {increase_ratio:.4f} (total: {total_labels}, class count: {class_labels[1]})")
    print(f"Normal Class Ratio: {normal_ratio:.4f} (total: {total_labels}, class count: {class_labels[0]})")

# Inspect the label ratios for all classes in the training directory
inspect_label_ratios(training_directory)

Processing chunk_0.pickle from val...
Processing chunk_1.pickle from val...
Processing chunk_10.pickle from val...
Processing chunk_100.pickle from val...
Processing chunk_101.pickle from val...
Processing chunk_102.pickle from val...
Processing chunk_103.pickle from val...
Processing chunk_104.pickle from val...
Processing chunk_105.pickle from val...
Processing chunk_106.pickle from val...
Processing chunk_107.pickle from val...
Processing chunk_108.pickle from val...
Processing chunk_109.pickle from val...
Processing chunk_11.pickle from val...
Processing chunk_110.pickle from val...
Processing chunk_111.pickle from val...
Processing chunk_112.pickle from val...
Processing chunk_113.pickle from val...
Processing chunk_114.pickle from val...
Processing chunk_115.pickle from val...
Processing chunk_116.pickle from val...
Processing chunk_117.pickle from val...
Processing chunk_118.pickle from val...
Processing chunk_119.pickle from val...
Processing chunk_12.pickle from val...
Process